# FinSight — The Full Flow (Phases 1 + 2 in one notebook)

One continuous tour from a ticker symbol to a **measured, cited answer**.
Each step runs for real; each links to the deep-dive that explains it slowly.

```
PHASE 1 (build the pipeline)                PHASE 2 (measure & improve it)
1 download   EDGAR -> 12.9MB HTML           6 golden dataset: 44 labelled questions
2 preprocess HTML -> clean text             7 keyword search + hybrid fusion (RRF)
3 sectionize text -> Item 1A, 7, ...        8 cross-encoder reranking
4 chunk      sections -> tagged passages    9 the eval: all modes, one table
5 embed+store -> Postgres/pgvector          10 cited answer + refusal, end to end
```

Deep dives: `phase1_deepdive.ipynb` (steps 1-5 spoon-fed) ·
`phase2_deepdive.ipynb` (steps 6-9 spoon-fed) ·
manual guides in `docs/*.html`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_colwidth", 90)
from IPython.display import Markdown, display
print("ready")

ready


## Step 1-2 · Download a filing and turn it into clean text
The SEC's EDGAR API is free (identify yourself, ~8 req/s). The filing arrives as
inline-XBRL HTML; `html_to_text` strips markup and flattens tables so numbers
keep their row labels.

In [2]:
from ingestion.edgar_client import EdgarClient
from ingestion.parser import html_to_text

edgar = EdgarClient()
filing = edgar.list_filings("JPM", forms=("10-K",), since="2024-01-01")[0]
raw = edgar.download_filing(filing).read_bytes()          # cached after 1st run
print(f"raw HTML: {len(raw)/1e6:.1f} MB")
text = html_to_text(raw)                                   # ~15s for 12.9 MB
print(f"clean text: {len(text):,} chars "
      f"(~{100*(1-len(text)/len(raw)):.0f}% of the file was markup)")

raw HTML: 12.9 MB


clean text: 1,449,219 chars (~89% of the file was markup)


## Step 3-4 · Split into named sections, then into tagged chunks
Sections follow the 10-K's standard "Item" structure (1A=Risk Factors, 7=MD&A).
Chunks are ~4,000 chars on paragraph boundaries, each prefixed with a metadata
header and carrying a ready-made citation.

In [3]:
from ingestion.parser import split_items
from ingestion.chunker import chunk_section

sections = split_items(text, "10-K")
risk = next(s for s in sections if s.item == "1A")
chunks = chunk_section(risk, "JPM", "10-K", filing.report_date)
print(f"{len(sections)} sections; Item 1A ({len(risk.text):,} chars) -> {len(chunks)} chunks")
print("\nA chunk's first lines:")
print(chunks[3].text[:260], "...")
print("\nits citation:", chunks[3].citation)

12 sections; Item 1A (112,633 chars) -> 24 chunks

A chunk's first lines:
[JPM | 10-K FY2025 | Item 1A: Risk Factors]
that are stricter than a global standard, which could create competitive disadvantages for those firms, such as JPMorganChase, that are subject to the enhanced regulations. Furthermore, certain authorities outside th ...

its citation: [JPM 10-K 2025, Item 1A]


## Step 5 · The vector store (already loaded)
`ingestion/pipeline.py` + `ingestion/embedder.py` did this for all 10 companies:
chunks -> `data/parsed/*.jsonl` -> 3072-dim embeddings -> rows in Postgres+pgvector.
One table holds metadata (exact SQL filters) AND vectors (similarity search).

In [4]:
import psycopg
from config import DATABASE_URL
with psycopg.connect(DATABASE_URL) as conn:
    rows = conn.execute(
        "SELECT ticker, count(*), min(period), max(period) FROM chunks GROUP BY ticker ORDER BY ticker"
    ).fetchall()
display(pd.DataFrame(rows, columns=["ticker", "chunks", "earliest 10-K", "latest 10-K"]))

,ticker,chunks,earliest 10-K,latest 10-K
0,AAPL,131,2024-09-28,2025-09-27
1,BAC,256,2025-12-31,2025-12-31
2,C,474,2025-12-31,2025-12-31
3,GOOGL,236,2024-12-31,2025-12-31
4,GS,419,2025-12-31,2025-12-31
5,JPM,475,2025-12-31,2025-12-31
6,MS,358,2025-12-31,2025-12-31
7,MSFT,243,2024-06-30,2025-06-30
8,NVDA,193,2025-01-26,2026-01-25
9,WFC,69,2024-12-31,2025-12-31


## Step 6 · The golden dataset — retrieval's labelled test set
44 questions; every answerable one pins verbatim `expected_phrases` verified to
exist in the corpus; 10 are deliberately unanswerable (the system must refuse).

In [5]:
golden = [json.loads(l) for l in open(ROOT/"evals/golden_dataset.jsonl", encoding="utf-8")]
display(pd.DataFrame([{"type": t, "n": sum(q["type"]==t for q in golden)}
    for t in ["single_fact","section","comparison","temporal","unanswerable"]]))
example = next(q for q in golden if q["type"]=="temporal")
print("example:", example["question"], "->", example["expected_phrases"])

,type,n
0,single_fact,18
1,section,6
2,comparison,5
3,temporal,6
4,unanswerable,10


example: Which iPhone models does Apple's FY2025 10-K list in the iPhone line? -> ['iPhone 17 Pro']


## Step 7-8 · Four retrieval modes
`vector` (Phase 1 baseline) · `keyword` (Postgres FTS/BM25-style) ·
`hybrid` (both, fused by reciprocal rank) · `hybrid+rerank`
(cross-encoder re-scores the top-24). One question, four answers:

In [6]:
from retrieval.search import retrieve
q = "Which iPhone models does Apple's FY2025 10-K list?"
for mode in ["vector", "keyword", "hybrid", "hybrid+rerank"]:
    hits = retrieve(q, k=3, mode=mode, tickers=["AAPL"])
    got = any("iphone 17 pro" in h.text.lower() for h in hits)
    print(f"{mode:14s} top citations {[h.citation for h in hits[:2]]}   phrase found: {got}")

vector         top citations ['[AAPL 10-K 2025, Item 1]', '[AAPL 10-K 2024, Item 1]']   phrase found: True
keyword        top citations ['[AAPL 10-K 2025, Item 1]', '[AAPL 10-K 2024, Item 1]']   phrase found: True


hybrid         top citations ['[AAPL 10-K 2025, Item 1]', '[AAPL 10-K 2024, Item 1]']   phrase found: True


C:\Users\subha\OneDrive\Desktop\Subham\finsight\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8830.02it/s]

hybrid+rerank  top citations ['[AAPL 10-K 2025, Item 1]', '[AAPL 10-K 2025, Item 7]']   phrase found: True


## Step 9 · The eval — every mode against the golden set
The Phase 2 deliverable: a measured comparison, not a vibe. (Takes a few
minutes; the rerank mode runs a CPU cross-encoder for every question.)

In [7]:
from evals.run_evals import eval_retrieval, load_golden
questions = load_golden()
table = []
for mode in ["vector", "keyword", "hybrid", "hybrid+rerank"]:
    r = eval_retrieval(mode, questions)
    table.append({"mode": mode, "hit rate": f"{r['hit_rate']:.0%}",
                  "citation acc (temporal)": f"{r['citation_acc']:.0%}",
                  "missed": len(r["misses"])})
    print(mode, "done")
display(pd.DataFrame(table))

vector done


keyword done


hybrid done


hybrid+rerank done


,mode,hit rate,citation acc (temporal),missed
0,vector,83%,100%,6
1,keyword,66%,100%,12
2,hybrid,80%,100%,7
3,hybrid+rerank,83%,100%,6


## Step 10 · End to end: a cited answer, and a refusal
The whole system in two questions — the strongest retrieval mode feeding the
cite-or-refuse prompt from Phase 1.

In [8]:
from rag_cli import SYSTEM_PROMPT
from ingestion.embedder import get_openai
from config import CHAT_DEPLOYMENT

aoai = get_openai()
def answer(question, tickers=None):
    hits = retrieve(question, k=8, mode="hybrid+rerank", tickers=tickers)
    evidence = "\n\n---\n\n".join(f"PASSAGE {i+1} {h.citation}:\n{h.text}"
                                  for i, h in enumerate(hits))
    return aoai.chat.completions.create(model=CHAT_DEPLOYMENT, messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"EVIDENCE:\n\n{evidence}\n\nQUESTION: {question}"},
    ]).choices[0].message.content

display(Markdown(answer("Compare how NVIDIA and Alphabet position themselves on AI.",
                        tickers=["NVDA", "GOOGL"])))

High-level: NVIDIA positions itself as a data‑center‑scale AI infrastructure company focused on accelerated computing across hardware, networking, and full‑stack software to reshape industries [NVDA 10-K 2026, Item 1]. Alphabet positions its AI offering around AI‑optimized cloud infrastructure, managed developer platforms, and foundation models and agents (Gemini) integrated into productivity tools [GOOGL 10-K 2025, Item 1].

NVIDIA emphasizes co‑designed, full‑stack infrastructure — GPUs, CPUs, DPUs, interconnects, networking, CUDA and other software libraries — and architectures such as Blackwell and Blackwell Ultra to enable large‑scale training and inference at data center scale [NVDA 10-K 2026, Item 1]. NVIDIA also highlights model platforms (e.g., Nemotron, Cosmos), enterprise software, and services including NVIDIA AI Enterprise and DGX Cloud for customers such as CSPs, AI model makers, enterprises and OEMs [NVDA 10-K 2026, Item 1] [NVDA 10-K 2025, Item 1].

Alphabet emphasizes delivering AI as cloud services: AI‑optimized infrastructure (TPUs and specialized GPUs), AI‑optimized storage, and a managed developer platform (Vertex AI) that gives access to hundreds of foundation models, including its Gemini family [GOOGL 10-K 2025, Item 1]. Alphabet also explicitly frames agents and workspace integrations (Gemini Enterprise; Gemini for Google Workspace) as ways to bring AI into business workflows and end‑user productivity [GOOGL 10-K 2025, Item 1].

Developer and ecosystem positioning differs: NVIDIA stresses its CUDA ecosystem and broad software/libraries to enable developers on its hardware and platforms [NVDA 10-K 2026, Item 1], while Alphabet emphasizes managed developer tools and platform access to many foundation models via Vertex AI and related services [GOOGL 10-K 2025, Item 1].

Net: NVIDIA foregrounds infrastructure, hardware acceleration, and full‑stack co‑design for large‑scale AI workloads; Alphabet foregrounds cloud‑delivered AI platforms, managed tooling, accelerator hardware in the cloud, and integrated foundation models/agents for developers and enterprise users [NVDA 10-K 2026, Item 1] [GOOGL 10-K 2025, Item 1].

In [9]:
display(Markdown(answer("What did Tesla say about vehicle recall risks?")))

Insufficient evidence in the indexed filings. Missing: Tesla’s SEC filings or provided passages that discuss Tesla’s statements regarding vehicle recall risks.

---
## Where the project goes next

- **Phase 3** — agents & orchestration: LangGraph supervisor routing, a quant
  agent over SEC XBRL (real numbers, never model memory), a critic agent that
  blocks uncited claims, human-in-the-loop on low confidence.
- **Phase 4** — productionization: FastAPI + streaming, Langfuse tracing,
  guardrails, Docker.
- **Phase 5** — Azure deployment with a CI eval gate: a deploy that regresses
  faithfulness never ships.